# 🔬 ASEAN Disease Surveillance AI — Fine-Tuning Pipeline (Google Colab)

Notebook ini digunakan untuk melatih (**fine-tune**) model **XLM-RoBERTa** secara mandiri menggunakan GPU Google Colab (T4 / A100).

### Fitur Continuous Learning:
1. **Human-in-the-loop Priority**: Data koreksi operator (`human_corrected`) otomatis diprioritaskan dengan bobot tertinggi (confidence 1.0).
2. **Multilingual Balance**: Menangani teks dalam Bahasa Indonesia, English, Vietnam, Thai, dan Melayu.
3. **Evaluasi Otomatis**: Menghitung Macro-F1, Precision, Recall, dan Confusion Matrix.
4. **Hot-Reload Ready**: Model output dapat langsung diekstrak ke folder server `/app/models/fine-tuned/` tanpa downtime.

## Langkah 1: Persiapan Environment & GPU Check

In [ ]:
# Cek ketersediaan GPU
!nvidia-smi

# Install dependencies terbaru
!pip install -q transformers[torch] datasets evaluate sentencepiece accelerate scikit-learn

## Langkah 2: Ambil Dataset dari Server / Upload File

Pilih salah satu cara:
- **Opsi A**: Masukkan URL API Server Anda (misal `https://surveillance.domain.com/nlp/export-dataset?limit=5000`)
- **Opsi B**: Upload file `train.jsonl` dan `test.jsonl` langsung ke panel Files di sebelah kiri Google Colab.

In [ ]:
import os, json, random
import urllib.request

# Masukkan URL API Server (Ganti sesuai domain server produksi Anda)
SERVER_API_URL = ""  # Contoh: "https://your-domain.com/nlp/export-dataset?limit=5000"

raw_data = []
if SERVER_API_URL:
    print(f"Mengunduh dataset dari {SERVER_API_URL}...")
    req = urllib.request.urlopen(SERVER_API_URL)
    res = json.loads(req.read().decode("utf-8"))
    raw_data = res.get("data", [])
    print(f"Berhasil mengunduh {len(raw_data)} contoh data!")
    print(f"Data hasil koreksi manusia: {res.get('human_corrected_count', 0)}")
else:
    print("SERVER_API_URL kosong. Memeriksa file upload lokal di Colab (train.jsonl / dataset.jsonl)...")
    for fname in ["train.jsonl", "dataset.jsonl", "dataset.json"]:
        if os.path.exists(fname):
            with open(fname, "r", encoding="utf-8") as f:
                if fname.endswith(".json"):
                    obj = json.load(f)
                    raw_data = obj.get("data", obj) if isinstance(obj, dict) else obj
                else:
                    raw_data = [json.loads(line) for line in f if line.strip()]
            print(f"Memuat {len(raw_data)} baris dari {fname}")
            break

assert len(raw_data) > 0, "Harap masukkan SERVER_API_URL atau upload file train.jsonl ke Google Colab!"


## Langkah 3: Preprocessing & Pembagian Train / Test Split

In [ ]:
from collections import Counter

# Filter contoh yang valid
valid_examples = []
for item in raw_data:
    text = (item.get("text") or "").strip()
    label = (item.get("disease_label") or item.get("disease") or "").strip()
    if len(text) > 20 and label and label.upper() not in {"UNKNOWN", "NEGATIVE"}:
        valid_examples.append({
            "text": text[:2000],
            "label": label,
            "source": item.get("source", "auto"),
            "confidence": float(item.get("confidence", 0.9))
        })

print(f"Total contoh valid: {len(valid_examples)}")

# Tampilkan distribusi label penyakit
label_counts = Counter(e["label"] for e in valid_examples)
print("\nDistribusi Label Penyakit:")
for lbl, cnt in label_counts.most_common(15):
    print(f"  - {lbl}: {cnt} contoh")

# Buat label mapping
unique_labels = sorted(list(label_counts.keys()))
label2id = {lbl: i for i, lbl in enumerate(unique_labels)}
id2label = {i: lbl for i, lbl in enumerate(unique_labels)}

# Split 85% Train, 15% Test
random.seed(42)
random.shuffle(valid_examples)
split_idx = int(len(valid_examples) * 0.85)
train_data = valid_examples[:split_idx]
test_data = valid_examples[split_idx:]

print(f"\nTrain set: {len(train_data)} | Test set: {len(test_data)}")


## Langkah 4: Load Base Model & Tokenizer (XLM-RoBERTa)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset

BASE_MODEL = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize_fn(batch):
    tokens = tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)
    tokens["labels"] = [label2id[l] for l in batch["label"]]
    return tokens

train_ds = Dataset.from_list(train_data).map(tokenize_fn, batched=True)
test_ds = Dataset.from_list(test_data).map(tokenize_fn, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id
)
print("Model XLM-RoBERTa berhasil dimuat!")


## Langkah 5: Training Execution (FP16 Accelerated)

In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

f1_metric = evaluate.load("f1")
acc_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    acc = acc_metric.compute(predictions=preds, references=labels)["accuracy"]
    return {"accuracy": acc, "macro_f1": macro_f1}

OUTPUT_DIR = "./fine_tuned_asean_disease"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("Memulai Fine-Tuning pada GPU...")
trainer.train()


## Langkah 6: Evaluasi Akhir & Export Weights

In [ ]:
# Evaluasi akhir pada test set
metrics = trainer.evaluate()
print("Hasil Evaluasi:", metrics)

# Simpan model final
FINAL_EXPORT = "./models/fine-tuned"
trainer.save_model(FINAL_EXPORT)
tokenizer.save_pretrained(FINAL_EXPORT)

# Simpan label manifest
with open(f"{FINAL_EXPORT}/labels.json", "w", encoding="utf-8") as f:
    json.dump({"id2label": id2label, "label2id": label2id, "metrics": metrics}, f, indent=2)

# Kompres menjadi zip/tar.gz untuk didownload
!tar -czvf fine_tuned_disease_model.tar.gz -C ./models/fine-tuned .

print("\n✅ Model siap didownload! File: fine_tuned_disease_model.tar.gz")
print("Cara pasang di server:")
print("1. Ekstrak isi file ke: services/nlp-python/models/fine-tuned/")
print("2. Panggil API: POST http://localhost:8000/reload")
print("3. Layanan NLP otomatis beralih menggunakan model baru tanpa restart!")
